# Trend Following Parameter Sweep Results

Visualization of sweep results from `sweep.py`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import PercentFormatter
from pathlib import Path

pd.set_option('display.max_rows', 100, 'display.max_columns', None)
RESULTS = Path('sweep_results')

In [ ]:
# Load all results
singles = pd.read_parquet(RESULTS / 'singles.parquet') if (RESULTS / 'singles.parquet').exists() else pd.DataFrame()
by_cat = pd.read_parquet(RESULTS / 'by_category.parquet') if (RESULTS / 'by_category.parquet').exists() else pd.DataFrame()
es_overlay = pd.read_parquet(RESULTS / 'es_overlay.parquet') if (RESULTS / 'es_overlay.parquet').exists() else pd.DataFrame()
blends = pd.read_parquet(RESULTS / 'blends.parquet') if (RESULTS / 'blends.parquet').exists() else pd.DataFrame()
bootstrap = pd.read_parquet(RESULTS / 'bootstrap.parquet') if (RESULTS / 'bootstrap.parquet').exists() else pd.DataFrame()
dense = pd.read_parquet(RESULTS / 'dense.parquet') if (RESULTS / 'dense.parquet').exists() else pd.DataFrame()
composite = pd.read_parquet(RESULTS / 'composite.parquet') if (RESULTS / 'composite.parquet').exists() else pd.DataFrame()
composite_pnl = pd.read_parquet(RESULTS / 'composite_pnl.parquet') if (RESULTS / 'composite_pnl.parquet').exists() else pd.DataFrame()

print(f'Singles: {len(singles)}, By category: {len(by_cat)}, ES overlay: {len(es_overlay)}')
print(f'Blends: {len(blends)}, Bootstrap: {len(bootstrap)}, Dense: {len(dense)}')
print(f'Composite: {len(composite)}')

## 1. Overview: Sharpe by signal type

In [ ]:
if len(singles) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    singles.boxplot(column='sharpe', by='signal_type', ax=ax)
    ax.set_title('Sharpe Distribution by Signal Type')
    ax.set_ylabel('Sharpe Ratio')
    plt.suptitle('')
    plt.tight_layout()

## 2. Top 20 single signals

In [ ]:
if len(singles) > 0:
    decade_cols = [c for c in singles.columns if c.startswith('sharpe_')]
    cat_cols = [c for c in singles.columns if c.startswith('cat_sharpe_')]
    display_cols = ['signal_type', 'vol_window', 'smoothing', 'param1', 'param2',
                    'sharpe', 'max_dd'] + decade_cols + cat_cols
    display(
        singles
        .sort_values('sharpe', ascending=False)
        .head(20)[display_cols]
        .reset_index(drop=True)
        .style.format('{:.3f}', subset=[c for c in display_cols if c not in ['signal_type', 'vol_window', 'smoothing', 'param1', 'param2']])
        .background_gradient(subset=['sharpe'], cmap='RdYlGn')
    )

## 3. Heatmaps: Sharpe vs parameters (per signal type)

In [ ]:
if len(singles) > 0:
    for stype in singles.signal_type.unique():
        sub = singles[singles.signal_type == stype]
        # Find best vol_window for this signal type
        best_vw = sub.groupby('vol_window').sharpe.median().idxmax()
        sub_vw = sub[sub.vol_window == best_vw]
        
        if 'param2' in sub_vw.columns and sub_vw.param2.nunique() > 1:
            # 2-param signal: heatmap of param1 vs param2 at best smoothing
            best_sm = sub_vw.groupby('smoothing').sharpe.median().idxmax()
            pivot = sub_vw[sub_vw.smoothing == best_sm].pivot_table(
                values='sharpe', index='param1', columns='param2', aggfunc='mean'
            )
            fig, ax = plt.subplots(figsize=(10, 6))
            im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn',
                          extent=[pivot.columns[0], pivot.columns[-1],
                                  pivot.index[-1], pivot.index[0]])
            ax.set_xlabel('param2 (slow)')
            ax.set_ylabel('param1 (fast)')
            ax.set_title(f'{stype} Sharpe (vol_window={best_vw}, smoothing={best_sm})')
            plt.colorbar(im, ax=ax, label='Sharpe')
        else:
            # 1-param signal: heatmap of param1 vs smoothing
            pivot = sub_vw.pivot_table(
                values='sharpe', index='param1', columns='smoothing', aggfunc='mean'
            )
            fig, ax = plt.subplots(figsize=(10, 6))
            im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn')
            ax.set_xticks(range(len(pivot.columns)))
            ax.set_xticklabels(pivot.columns)
            ax.set_yticks(range(len(pivot.index)))
            ax.set_yticklabels(pivot.index)
            ax.set_xlabel('Smoothing')
            ax.set_ylabel('Lookback / Window')
            ax.set_title(f'{stype} Sharpe (vol_window={best_vw})')
            plt.colorbar(im, ax=ax, label='Sharpe')
        plt.tight_layout()
        plt.show()

## 4. Volatility window effect

In [ ]:
if len(singles) > 0:
    fig, ax = plt.subplots(figsize=(8, 5))
    singles.boxplot(column='sharpe', by='vol_window', ax=ax)
    ax.set_title('Sharpe by Volatility Estimation Window')
    ax.set_ylabel('Sharpe')
    plt.suptitle('')
    plt.tight_layout()

## 5. Per-category best signals

In [ ]:
if len(by_cat) > 0:
    best_per_cat = (
        by_cat
        .sort_values('sharpe', ascending=False)
        .groupby('category').first()
        .sort_values('sharpe', ascending=False)
    )
    display(best_per_cat[['signal_type', 'vol_window', 'smoothing', 'param1', 'param2', 'sharpe', 'max_dd']])
    
    # Heatmap per category
    for cat in by_cat.category.unique():
        cat_df = by_cat[by_cat.category == cat]
        fig, ax = plt.subplots(figsize=(10, 5))
        cat_df.boxplot(column='sharpe', by='signal_type', ax=ax)
        ax.set_title(f'{cat}: Sharpe by Signal Type')
        ax.set_ylabel('Sharpe')
        plt.suptitle('')
        plt.tight_layout()
        plt.show()

## 6. Composite portfolio (best per category)

In [ ]:
if len(composite) > 0:
    display(composite)

if len(composite_pnl) > 0:
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    
    # Equity curve
    ax = axes[0]
    composite_pnl.cumsum().plot(ax=ax, lw=1)
    ax.set_title('Composite Portfolio - Cumulative Returns')
    ax.set_ylabel('Cumulative vol-normalized points')
    ax.legend(loc='upper left')
    
    # Drawdown
    ax = axes[1]
    cum = composite_pnl['composite'].cumsum()
    dd = cum - cum.cummax()
    dd.plot(ax=ax, color='red', lw=1)
    ax.fill_between(dd.index, dd, 0, alpha=0.3, color='red')
    ax.set_title('Composite Drawdown')
    ax.set_ylabel('Drawdown')
    
    plt.tight_layout()

## 7. ES overlay

In [ ]:
if len(es_overlay) > 0:
    print(f'Best combined Sharpe: {es_overlay.combined_sharpe.max():.3f}')
    print(f'Best ES-only Sharpe: {es_overlay.es_sharpe.max():.3f}')
    
    best_vw = es_overlay.groupby('vol_window').combined_sharpe.median().idxmax()
    sub = es_overlay[es_overlay.vol_window == best_vw]
    
    pivot = sub.pivot_table(values='combined_sharpe', index='lookback', columns='smoothing')
    fig, ax = plt.subplots(figsize=(10, 6))
    im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel('Smoothing')
    ax.set_ylabel('Lookback')
    ax.set_title(f'ES Overlay: Combined Sharpe (vol_window={best_vw})')
    plt.colorbar(im, ax=ax, label='Combined Sharpe')
    plt.tight_layout()

## 8. Blend analysis

In [ ]:
if len(blends) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    ax = axes[0]
    best_component = blends[['sharpe_a', 'sharpe_b']].max(axis=1)
    ax.scatter(best_component, blends.blend_sharpe, alpha=0.3, s=5)
    lims = [best_component.min(), best_component.max()]
    ax.plot(lims, lims, 'r--', lw=1)
    ax.set_xlabel('Best Component Sharpe')
    ax.set_ylabel('Blend Sharpe')
    ax.set_title('Blend vs Best Component')
    
    ax = axes[1]
    blends.improvement.hist(bins=50, ax=ax)
    ax.axvline(0, color='red', lw=1)
    ax.set_xlabel('Improvement over best component')
    ax.set_title('Blend Improvement Distribution')
    
    plt.tight_layout()
    
    print('Top 10 blends:')
    display(
        blends.sort_values('blend_sharpe', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

## 9. Bootstrap confidence intervals

In [ ]:
if len(bootstrap) > 0:
    boot = bootstrap.sort_values('sharpe', ascending=False).head(30).reset_index(drop=True)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    labels = boot.apply(lambda r: f"{r.signal_type}\n({int(r.param1)},{int(r.smoothing)})", axis=1)
    y = range(len(boot))
    
    ax.barh(y, boot.sharpe, height=0.4, color='steelblue', alpha=0.7, label='Point estimate')
    ax.errorbar(boot.boot_50pct, y, 
                xerr=[boot.boot_50pct - boot.boot_5pct, boot.boot_95pct - boot.boot_50pct],
                fmt='none', color='black', capsize=3, label='90% CI')
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xlabel('Sharpe Ratio')
    ax.set_title('Top 30 Signals: Bootstrap Confidence Intervals')
    ax.legend()
    ax.invert_yaxis()
    plt.tight_layout()

## 10. Dense grid (fine-tuning around best regions)

In [ ]:
if len(dense) > 0:
    for stype in dense.signal_type.unique():
        sub = dense[dense.signal_type == stype]
        pivot = sub.pivot_table(values='sharpe', index='param1', columns='smoothing', aggfunc='mean')
        fig, ax = plt.subplots(figsize=(12, 6))
        im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn')
        # Label every 5th tick
        ax.set_xticks(range(0, len(pivot.columns), max(1, len(pivot.columns)//10)))
        ax.set_xticklabels([pivot.columns[i] for i in range(0, len(pivot.columns), max(1, len(pivot.columns)//10))])
        ax.set_yticks(range(0, len(pivot.index), max(1, len(pivot.index)//10)))
        ax.set_yticklabels([pivot.index[i] for i in range(0, len(pivot.index), max(1, len(pivot.index)//10))])
        ax.set_xlabel('Smoothing')
        ax.set_ylabel('Param1')
        ax.set_title(f'{stype} Dense Grid - Sharpe')
        plt.colorbar(im, ax=ax, label='Sharpe')
        plt.tight_layout()
        plt.show()
    
    print('\nDense grid top 10:')
    display(
        dense.sort_values('sharpe', ascending=False)
        .head(10)[['signal_type', 'vol_window', 'smoothing', 'param1', 'param2', 'sharpe', 'max_dd']]
        .reset_index(drop=True)
    )

## 11. Decade stability

In [ ]:
if len(singles) > 0:
    decade_cols = [c for c in singles.columns if c.startswith('sharpe_') and not c.startswith('sharpe_stability')]
    top20 = singles.sort_values('sharpe', ascending=False).head(20)
    
    fig, ax = plt.subplots(figsize=(14, 8))
    data = top20[decade_cols].values
    im = ax.imshow(data, aspect='auto', cmap='RdYlGn', vmin=-1, vmax=2)
    ax.set_xticks(range(len(decade_cols)))
    ax.set_xticklabels([c.replace('sharpe_', '') for c in decade_cols])
    labels = top20.apply(lambda r: f"{r.signal_type} ({int(r.param1)}, sm={int(r.smoothing)}, vw={int(r.vol_window)})", axis=1)
    ax.set_yticks(range(len(top20)))
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_title('Top 20 Signals: Per-Decade Sharpe')
    plt.colorbar(im, ax=ax, label='Sharpe')
    plt.tight_layout()

## 12. Summary statistics

In [ ]:
if len(singles) > 0:
    summary = (
        singles
        .groupby('signal_type')
        .agg(
            count=('sharpe', 'count'),
            sharpe_mean=('sharpe', 'mean'),
            sharpe_median=('sharpe', 'median'),
            sharpe_max=('sharpe', 'max'),
            sharpe_std=('sharpe', 'std'),
            max_dd_median=('max_dd', 'median'),
        )
        .sort_values('sharpe_median', ascending=False)
    )
    display(summary.style.format('{:.3f}', subset=summary.columns[1:]))